# Data Audit and Cleaning

In [ ]:
pip install pandas numpy matplotlib seaborn scikit-learn

  Using cached scikit_learn-1.9.1-cp313-cp313-win_amd64.whl.metadata (9.3 kB)
  Using cached scipy-1.18.1-cp313-cp313-win_amd64.whl.metadata (61 kB)
  Using cached joblib-1.6.0-py3-none-any.whl.metadata (6.5 kB)
  Using cached threadpoolctl-3.8.0-py3-none-any.whl.metadata (29 kB)
  Using cached cloudpickle-3.1.3-py3-none-any.whl.metadata (7.1 kB)
Using cached scikit_learn-1.9.1-cp313-cp313-win_amd64.whl (8.2 MB)
Using cached joblib-1.6.0-py3-none-any.whl (306 kB)
Using cached cloudpickle-3.1.3-py3-none-any.whl (22 kB)
Using cached scipy-1.18.1-cp313-cp313-win_amd64.whl (36.6 MB)
Using cached threadpoolctl-3.8.0-py3-none-any.whl (32 kB)

   ------ --------------------------------- 1/6 [scipy]
   ------ --------------------------------- 1/6 [scipy]
   ------ --------------------------------- 1/6 [scipy]
   ------ --------------------------------- 1/6 [scipy]
   ------ --------------------------------- 1/6 [scipy]
   ------ --------------------------------- 1/6 [scipy]
   ------ ---------

In [ ]:
pip freeze > requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import pandas as pd
import numpy as np
import warnings
import lightgbm as lgb

from sklearn.covariance import LedoitWolf
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from scipy.linalg import inv

warnings.filterwarnings("ignore")

### Target Columns

In [ ]:
REGIONS = ["nr", "wr", "sr", "er", "ner"]

# Regional renewable, hydro, consumption, and net inter-regional exchange targets
REG_HYDRO      = [f"hydro_gen_{r}_mu" for r in REGIONS]
REG_SOLAR      = [f"solar_gen_{r}_mu" for r in REGIONS]
REG_WIND       = ["wind_gen_nr_mu", "wind_gen_wr_mu", "wind_gen_sr_mu"]  # ER & NER wind are 0
REG_ENERGY_MET = [f"energy_met_{r}_mu" for r in REGIONS]
REG_IE_ACTUAL  = [f"ie_actual_{r}_mu" for r in REGIONS]

# National source-wise targets
NATIONAL_SOURCES = [
    "gen_coal_mu",
    "gen_lignite_mu",
    "gen_hydro_mu",
    "gen_nuclear_mu",
    "gen_gas_mu",
    "gen_res_mu",
]

# Top-level national totals
NATIONAL_TOTALS = [
    "gen_total_mu",
    "solar_gen_total_mu",
    "wind_gen_total_mu",
    "energy_met_total_mu",
    "ie_actual_total_mu", 
]

ALL_TARGETS = (
    REG_HYDRO + REG_SOLAR + REG_WIND + REG_ENERGY_MET
    + REG_IE_ACTUAL + NATIONAL_SOURCES + NATIONAL_TOTALS
)

# Route targets based on your 6-fold benchmark: use Persistence where MASE >= 1.0
PERSISTENCE_TARGETS = {
    "gen_nuclear_mu",
    "gen_gas_mu",
    "gen_lignite_mu",
    "solar_gen_total_mu",
    "solar_gen_nr_mu",
    "solar_gen_wr_mu",
    "solar_gen_sr_mu",
    "solar_gen_er_mu",
    "solar_gen_ner_mu",
}

print("Total targets to predict:", len(ALL_TARGETS))

Total targets to predict: 34


### Load Data set and fix the missing dates

In [ ]:
csv_path = r"C:\Users\paris\Downloads\archive (4)\study1_daily.csv"

df = pd.read_csv(csv_path)
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").drop_duplicates("date").reset_index(drop=True)

# Create continuous daily calendar
full_dates = pd.date_range(start=df["date"].min(), end=df["date"].max(), freq="D")
df = df.set_index("date").reindex(full_dates).rename_axis("date").reset_index()

# Fix ER and NER wind generation (structurally zero)
df["wind_gen_er_mu"] = df["wind_gen_er_mu"].fillna(0.0)
df["wind_gen_ner_mu"] = df["wind_gen_ner_mu"].fillna(0.0)

# Flag the 2023 NLDC reporting definition shift
df["is_post_2023"] = (df["date"] >= "2023-01-01").astype(int)

print("Dataset shape:", df.shape)
print("Date range:", df["date"].min().strftime("%Y-%m-%d"), "to", df["date"].max().strftime("%Y-%m-%d"))

Dataset shape: (2884, 145)
Date range: 2018-12-31 to 2026-11-22


### Calender features

In [ ]:
target_date = df["date"] + pd.Timedelta(days=1)

df["target_month"] = target_date.dt.month
df["target_dayofweek"] = target_date.dt.dayofweek
df["target_is_weekend"] = (target_date.dt.dayofweek >= 5).astype(int)

day_of_year = target_date.dt.dayofyear

df["target_sin_doy"] = np.sin(
    2 * np.pi * day_of_year / 365.25
)

df["target_cos_doy"] = np.cos(
    2 * np.pi * day_of_year / 365.25
)

df

,date,evening_peak_demand_nr_mw,evening_peak_demand_wr_mw,evening_peak_demand_sr_mw,evening_peak_demand_er_mw,evening_peak_demand_ner_mw,evening_peak_demand_total_mw,peak_shortage_nr_mw,peak_shortage_wr_mw,peak_shortage_sr_mw,...,xb_net_bhutan_mu,xb_net_nepal_mu,xb_net_bangladesh_mu,xb_net_myanmar_mu,is_post_2023,target_month,target_dayofweek,target_is_weekend,target_sin_doy,target_cos_doy
0,2018-12-31,45890.0,46715.0,41005.0,18304.0,2396.0,154310.0,727.0,0.0,0.0,...,NaN,NaN,NaN,NaN,0,1,1,0,0.017202,0.999852
1,2019-01-01,45737.0,45720.0,38540.0,17423.0,2314.0,149734.0,609.0,0.0,0.0,...,NaN,NaN,NaN,NaN,0,1,2,0,0.034398,0.999408
2,2019-01-02,45781.0,47429.0,41507.0,17744.0,2427.0,154888.0,704.0,0.0,0.0,...,NaN,NaN,NaN,NaN,0,1,3,0,0.051584,0.998669
3,2019-01-03,45758.0,47824.0,41563.0,16750.0,2432.0,154327.0,1203.0,0.0,33.0,...,NaN,NaN,NaN,NaN,0,1,4,0,0.068755,0.997634
4,2019-01-04,46268.0,47440.0,41363.0,17169.0,2439.0,154679.0,1371.0,0.0,0.0,...,NaN,NaN,NaN,NaN,0,1,5,1,0.085906,0.996303
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2879,2026-11-18,60413.0,66624.0,55102.0,23201.0,2990.0,208330.0,0.0,0.0,0.0,...,-4.50,11.34,-23.47,0.0,1,11,3,0,-0.664483,0.747303
2880,2026-11-19,59951.0,66458.0,55072.0,22840.0,2959.0,207280.0,0.0,0.0,0.0,...,-5.14,11.65,-23.57,0.0,1,11,4,0,-0.651530,0.758623
2881,2026-11-20,59302.0,64819.0,55824.0,22410.0,2945.0,205300.0,315.0,0.0,0.0,...,-6.10,12.65,-23.62,0.0,1,11,5,1,-0.638384,0.769718
2882,2026-11-21,58760.0,66865.0,55008.0,22904.0,2961.0,206498.0,464.0,0.0,0.0,...,-6.25,14.88,-23.62,0.0,1,11,6,1,-0.625050,0.780585


### Domain Driver Features

In [ ]:
driver_cols = [
    f"energy_met_{r}_mu" for r in REGIONS
] + [
    f"outage_total_{r}_mw" for r in REGIONS
] + [
    f"ie_actual_{r}_mu" for r in REGIONS
] + [
    "energy_met_total_mu",
    "outage_total_total_mw",
    "freq_fvi",
    "gen_total_mu",
    "share_res_pct",
]

driver_cols = [c for c in driver_cols if c in df.columns]

calendar_cols = [
    "is_post_2023",
    "target_month",
    "target_dayofweek",
    "target_is_weekend",
    "target_sin_doy",
    "target_cos_doy",
]

feature_cols = list(dict.fromkeys(calendar_cols + driver_cols))
print("Base feature count:", len(feature_cols))

Base feature count: 26


### Lag and rolling features

In [ ]:
for col in ALL_TARGETS:

    # Today's generation value
    feature_cols.append(col)

    # Lag features
    df[f"{col}_lag1"] = df[col].shift(1)
    df[f"{col}_lag6"] = df[col].shift(6)

    # Rolling averages
    df[f"{col}_roll7_mean"] = (
        df[col].rolling(window=7, min_periods=1).mean()
    )

    df[f"{col}_roll14_mean"] = (
        df[col].rolling(window=14, min_periods=1).mean()
    )

    feature_cols.extend([
        f"{col}_lag1",
        f"{col}_lag6",
        f"{col}_roll7_mean",
        f"{col}_roll14_mean",
    ])

# print("Total feature count:", len(feature_cols))

# Remove any accidental duplicate column names
feature_cols = list(dict.fromkeys(feature_cols))
print("Total feature count:", len(feature_cols))

Total feature count: 184


### Identify today's row and tomorrow's forecast date

In [ ]:
latest_today_row = df.iloc[[-1]].copy()

latest_date = latest_today_row["date"].dt.strftime("%Y-%m-%d").values[0]

forecast_date = (
    latest_today_row["date"] + pd.Timedelta(days=1)
).dt.strftime("%Y-%m-%d").values[0]

print("Latest data date :", latest_date)
print("Forecasting for  :", forecast_date)

Latest data date : 2026-11-22
Forecasting for  : 2026-11-23


### Initialize output storage

In [ ]:
predictions = {}
pred_intervals = {}  # Stores {"q10": lower, "q90": upper} for 80% prediction intervals
validation_scores = {}
val_records = []

train_ratio = 0.80

### Train, validation, and prediction

In [ ]:
lgb_params = dict(
    n_estimators=250,
    learning_rate=0.02,
    num_leaves=15,
    min_child_samples=30,
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.7,
    reg_alpha=5.0,
    reg_lambda=10.0,
    random_state=42,
    verbose=-1,
)

for target in ALL_TARGETS:
    y_diff = df[target].shift(-1) - df[target]
    y_tomorrow_actual = df[target].shift(-1)
    target_dates = df["date"] + pd.Timedelta(days=1)

    valid_mask = y_diff.notnull() & df[target].notnull()

    X = df.loc[valid_mask, feature_cols]
    y = y_diff.loc[valid_mask]
    y_actual_level = y_tomorrow_actual.loc[valid_mask]
    today_level = df.loc[valid_mask, target]
    dates_valid = target_dates.loc[valid_mask]

    # Strict chronological 80 / 20 split
    split_idx = int(len(X) * train_ratio)

    X_train, y_train = X.iloc[:split_idx], y.iloc[:split_idx]
    X_test, y_test_actual = X.iloc[split_idx:], y_actual_level.iloc[split_idx:]
    print(X_train.columns)
    print(y_train.columns)
    print(X_test.columns)
    print(y_test_actual.columns)
    today_test = today_level.iloc[split_idx:]
    dates_test = dates_valid.iloc[split_idx:]

    # Allow negative values only for net inter-regional exchange (exporters are negative)
    allow_neg = target.startswith("ie_actual_")

    if target not in PERSISTENCE_TARGETS:
        model = lgb.LGBMRegressor(objective="quantile", alpha=0.50, **lgb_params)
        model.fit(X_train, y_train)
        pred_diff = model.predict(X_test)
        test_preds = today_test.values + pred_diff
    else:
        # Persistence point forecast (tomorrow = today)
        test_preds = today_test.values.copy()

    if not allow_neg:
        test_preds = np.clip(test_preds, 0, None)

    mae = mean_absolute_error(y_test_actual, test_preds)
    validation_scores[target] = round(mae, 2)

    target_test_df = pd.DataFrame({
        "Target_Date": dates_test.dt.strftime("%Y-%m-%d").values,
        "Target_Source": target,
        "Actual_MU": np.round(y_test_actual.values, 2),
        "Predicted_MU": np.round(test_preds, 2),
        "Error_MU": np.round(y_test_actual.values - test_preds, 2),
        "Abs_Error_MU": np.round(np.abs(y_test_actual.values - test_preds), 2),
    })
    val_records.append(target_test_df)

    # Retrain on 100% of historical data to predict tomorrow + 80% Prediction Intervals
    today_val = float(latest_today_row[target].values[0])

    if target not in PERSISTENCE_TARGETS:
        q_vals = {}
        for q, label in [(0.10, "q10"), (0.50, "q50"), (0.90, "q90")]:
            m_q = lgb.LGBMRegressor(objective="quantile", alpha=q, **lgb_params)
            m_q.fit(X, y)
            val = today_val + float(m_q.predict(latest_today_row[feature_cols])[0])
            q_vals[label] = val if allow_neg else max(0.0, val)

        q_vals["q10"] = min(q_vals["q10"], q_vals["q50"])
        q_vals["q90"] = max(q_vals["q90"], q_vals["q50"])

        predictions[target] = round(q_vals["q50"], 2)
        pred_intervals[target] = {"q10": round(q_vals["q10"], 2), "q90": round(q_vals["q90"], 2)}
    else:
        # Persistence point forecast + empirical 10th/90th percentile daily changes from post-2023
        post23_diffs = y[df.loc[valid_mask, "is_post_2023"] == 1]
        d10, d90 = np.nanpercentile(post23_diffs, [10, 90])
        p50 = today_val if allow_neg else max(0.0, today_val)
        p10 = (today_val + d10) if allow_neg else max(0.0, today_val + d10)
        p90 = (today_val + d90) if allow_neg else max(0.0, today_val + d90)

        predictions[target] = round(p50, 2)
        pred_intervals[target] = {"q10": round(p10, 2), "q90": round(p90, 2)}

# Structural zeros for ER and NER wind
for zero_col in ["wind_gen_er_mu", "wind_gen_ner_mu"]:
    predictions[zero_col] = 0.0
    pred_intervals[zero_col] = {"q10": 0.0, "q90": 0.0}
    validation_scores[zero_col] = 0.0

val_results = pd.concat(val_records, ignore_index=True)
print("80/20 Routed Training & Quantile Evaluation Complete!")

80/20 Routed Training & Quantile Evaluation Complete!


### Reconcile Predictions So All Totals Match

In [ ]:
# ==========================================
# RECONCILE PREDICTIONS SO ALL TOTALS MATCH
# ==========================================
def reconcile_group(top_col, bottom_cols, preds_dict, intervals_dict, val_df, allow_negative=False):
    """MinT-Shrink reconciliation for a 2-level hierarchy (top + children move together)."""
    cols = [top_col] + bottom_cols
    y_hat = np.array([preds_dict[c] for c in cols], dtype=float)

    K = len(bottom_cols)
    S = np.vstack([np.ones((1, K)), np.eye(K)])

    err_piv = val_df[val_df["Target_Source"].isin(cols)].pivot(
        index="Target_Date", columns="Target_Source", values="Error_MU"
    )[cols].dropna()

    W = LedoitWolf().fit(err_piv.values).covariance_
    W_inv = inv(W)
    P = S @ inv(S.T @ W_inv @ S) @ S.T @ W_inv
    y_reconciled = P @ y_hat
    if not allow_negative:
        y_reconciled = np.clip(y_reconciled, 0, None)

    for col, val in zip(cols, y_reconciled):
        new_val = round(float(val), 2)
        delta = new_val - preds_dict[col]
        preds_dict[col] = new_val
        if col in intervals_dict:
            for q in ("q10", "q90"):
                x = intervals_dict[col][q] + delta
                intervals_dict[col][q] = round(x if allow_negative else max(0.0, x), 2)


def reconcile_children_to_total(children, preds, intervals, val_df, total_value, allow_negative=False):
    """Adjust children only so they sum to a FIXED total (GLS weights from validation errors)."""
    y = np.array([preds[c] for c in children], dtype=float)
    E = val_df[val_df["Target_Source"].isin(children)].pivot(
        index="Target_Date", columns="Target_Source", values="Error_MU"
    )[children].dropna()

    W = LedoitWolf().fit(E.values).covariance_
    one = np.ones(len(children))
    adj = y + W @ one * (total_value - y.sum()) / (one @ W @ one)

    if not allow_negative:
        for _ in range(10):
            adj = np.clip(adj, 0, None)
            gap = total_value - adj.sum()
            if abs(gap) < 1e-9:
                break
            free = adj > 0
            if not np.any(free):
                break
            w = W[np.ix_(free, free)] @ np.ones(free.sum())
            adj[free] += w * gap / w.sum()

    for c, v in zip(children, adj):
        new = round(float(v), 2)
        delta = new - preds[c]
        preds[c] = new
        if c in intervals:
            for q in ("q10", "q90"):
                x = intervals[c][q] + delta
                intervals[c][q] = round(x if allow_negative else max(0.0, x), 2)


reconciled_preds = predictions.copy()
reconciled_intervals = {k: v.copy() for k, v in pred_intervals.items()}

# 1. National sources -> total (top of the tree, locked once)
reconcile_group("gen_total_mu", NATIONAL_SOURCES, reconciled_preds, reconciled_intervals, val_results)

# 2. Regions -> FIXED national hydro (only regional hydro values move)
reconcile_children_to_total(
    REG_HYDRO,
    reconciled_preds,
    reconciled_intervals,
    val_results,
    total_value=reconciled_preds["gen_hydro_mu"],
)

# 3. Independent hierarchies (nothing else touches these)
reconcile_group("solar_gen_total_mu", REG_SOLAR, reconciled_preds, reconciled_intervals, val_results)
reconcile_group("wind_gen_total_mu", REG_WIND, reconciled_preds, reconciled_intervals, val_results)
reconcile_group("energy_met_total_mu", REG_ENERGY_MET, reconciled_preds, reconciled_intervals, val_results)
reconcile_group("ie_actual_total_mu", REG_IE_ACTUAL, reconciled_preds, reconciled_intervals, val_results, allow_negative=True)

# 4. RES physical consistency check: warn instead of editing so national total stays intact
if reconciled_preds["gen_res_mu"] < (reconciled_preds["solar_gen_total_mu"] + reconciled_preds["wind_gen_total_mu"]):
    print("WARNING: gen_res_mu < solar + wind for", forecast_date)

# 5. Verify exact sum checks
source_sum = sum(reconciled_preds[c] for c in NATIONAL_SOURCES)
hydro_sum  = sum(reconciled_preds[c] for c in REG_HYDRO)

print(f"Reconciled Total Generation ({forecast_date}): {reconciled_preds['gen_total_mu']:.2f} MU | "
      f"Sum of 6 Sources: {source_sum:.2f} MU")
print(f"Reconciled National Hydro   ({forecast_date}): {reconciled_preds['gen_hydro_mu']:.2f} MU | "
      f"Sum of 5 Regions: {hydro_sum:.2f} MU")

Reconciled Total Generation (2026-11-23): 5023.09 MU | Sum of 6 Sources: 5023.10 MU
Reconciled National Hydro   (2026-11-23): 260.03 MU | Sum of 5 Regions: 260.03 MU


### Calculate Total Region-Wise Generation (Including Losses)

In [ ]:
# ==========================================
# CALCULATE REGION-WISE GENERATION (EXCHANGE-ADJUSTED, NO ZERO-SUM FORCING)
# ==========================================
# Use reconciled regional net exchanges as-is (preserves real net cross-border export balance)
ie_raw = np.array([reconciled_preds[f"ie_actual_{r}_mu"] for r in REGIONS], dtype=float)

# Step 1: Compute net regional generation before losses/auxiliary consumption
regional_net_gen = {}
for idx, r in enumerate(REGIONS):
    em_r = reconciled_preds[f"energy_met_{r}_mu"]
    net_g = em_r - ie_raw[idx]  # Subtracts net imports (adds back net exports)
    regional_net_gen[f"net_gen_{r}_mu"] = round(net_g, 2)

# Step 2: Scale by gen_total_mu / sum(net_gen) so regional gross generation sums identically to gen_total_mu
sum_net_gen = sum(regional_net_gen.values())
gross_to_net = reconciled_preds["gen_total_mu"] / sum_net_gen

regional_gross_gen = {}
for r in REGIONS:
    regional_gross_gen[f"total_gen_{r}_mu"] = round(regional_net_gen[f"net_gen_{r}_mu"] * gross_to_net, 2)

print(f"\n=== Exchange-Adjusted Region-Wise Generation Forecast ({forecast_date}) ===")
print(f"Total Net IR Exchange Sum : {ie_raw.sum():>8.2f} MU")
print(f"Gross-to-Net Multiplier   : {gross_to_net:>8.4f}\n")

for r in REGIONS:
    print(f"{r.upper():<5} | Net Gen: {regional_net_gen[f'net_gen_{r}_mu']:>8.2f} MU | "
          f"Gross Gen: {regional_gross_gen[f'total_gen_{r}_mu']:>8.2f} MU")

print("-" * 52)
print(f"{'SUM':<5} | Net Gen: {sum_net_gen:>8.2f} MU | "
      f"Gross Gen: {sum(regional_gross_gen.values()):>8.2f} MU (Target: {reconciled_preds['gen_total_mu']:.2f} MU)")


=== Exchange-Adjusted Region-Wise Generation Forecast (2026-11-23) ===
Total Net IR Exchange Sum :   -13.65 MU
Gross-to-Net Multiplier   :   1.0875

NR    | Net Gen:  1082.95 MU | Gross Gen:  1177.68 MU
WR    | Net Gen:  1791.61 MU | Gross Gen:  1948.32 MU
SR    | Net Gen:   982.69 MU | Gross Gen:  1068.65 MU
ER    | Net Gen:   714.34 MU | Gross Gen:   776.82 MU
NER   | Net Gen:    47.47 MU | Gross Gen:    51.62 MU
----------------------------------------------------
SUM   | Net Gen:  4619.06 MU | Gross Gen:  5023.09 MU (Target: 5023.09 MU)


### Final Forecast Tables

In [ ]:
# Table 1: National Total & Source-Wise Forecast with 80% Prediction Intervals
nat_cols = ["gen_total_mu"] + NATIONAL_SOURCES
national_table = pd.DataFrame({
    "Source": nat_cols,
    f"Forecast_{forecast_date}_MU": [reconciled_preds[c] for c in nat_cols],
    "Lower_80pct_MU": [reconciled_intervals[c]["q10"] for c in nat_cols],
    "Upper_80pct_MU": [reconciled_intervals[c]["q90"] for c in nat_cols],
    "Test_20pct_MAE_MU": [validation_scores[c] for c in nat_cols],
})

# Table 2: Exchange-Adjusted Region-Wise Total Generation Forecast
region_total_table = pd.DataFrame({
    "Region": [r.upper() for r in REGIONS],
    f"Energy_Met_{forecast_date}_MU": [reconciled_preds[f"energy_met_{r}_mu"] for r in REGIONS],
    "Est_Net_IR_Import_MU": [reconciled_preds[f"ie_actual_{r}_mu"] for r in REGIONS],
    f"Est_Net_Gen_{forecast_date}_MU": [regional_net_gen[f"net_gen_{r}_mu"] for r in REGIONS],
    f"Est_Gross_Gen_{forecast_date}_MU": [regional_gross_gen[f"total_gen_{r}_mu"] for r in REGIONS],
})

# Table 3: Region-Wise Renewable & Hydro Breakdown
region_source_table = pd.DataFrame({
    "Region": [r.upper() for r in REGIONS],
    "Hydro_MU": [reconciled_preds[f"hydro_gen_{r}_mu"] for r in REGIONS],
    "Solar_MU": [reconciled_preds[f"solar_gen_{r}_mu"] for r in REGIONS],
    "Wind_MU":  [reconciled_preds[f"wind_gen_{r}_mu"]  for r in REGIONS],
})

print("=== 1. NATIONAL TOTAL & SOURCE-WISE GENERATION FORECAST (WITH 80% INTERVALS) ===")
display(national_table)

print("\n=== 2. EXCHANGE-ADJUSTED REGION-WISE TOTAL GENERATION FORECAST ===")
display(region_total_table)

print("\n=== 3. REGION-WISE HYDRO, SOLAR & WIND FORECAST ===")
display(region_source_table)

=== 1. NATIONAL TOTAL & SOURCE-WISE GENERATION FORECAST (WITH 80% INTERVALS) ===


,Source,Forecast_2026-11-23_MU,Lower_80pct_MU,Upper_80pct_MU,Test_20pct_MAE_MU
0,gen_total_mu,5023.09,4830.28,5080.27,79.76
1,gen_coal_mu,3611.08,3434.01,3668.47,73.11
2,gen_lignite_mu,90.90,84.96,96.77,3.65
3,gen_hydro_mu,260.03,241.69,280.37,16.54
4,gen_nuclear_mu,174.83,169.84,179.67,2.69
5,gen_gas_mu,38.13,23.48,54.53,10.21
6,gen_res_mu,848.13,794.84,916.94,48.30



=== 2. EXCHANGE-ADJUSTED REGION-WISE TOTAL GENERATION FORECAST ===


,Region,Energy_Met_2026-11-23_MU,Est_Net_IR_Import_MU,Est_Net_Gen_2026-11-23_MU,Est_Gross_Gen_2026-11-23_MU
0,NR,1202.23,119.28,1082.95,1177.68
1,WR,1551.96,-239.65,1791.61,1948.32
2,SR,1293.27,310.58,982.69,1068.65
3,ER,504.60,-209.74,714.34,776.82
4,NER,53.35,5.88,47.47,51.62



=== 3. REGION-WISE HYDRO, SOLAR & WIND FORECAST ===


,Region,Hydro_MU,Solar_MU,Wind_MU
0,NR,126.71,223.04,18.27
1,WR,43.18,217.73,95.93
2,SR,50.28,180.46,46.21
3,ER,21.43,3.43,0.00
4,NER,18.43,0.47,0.00


### Export Results to CSV (Keep As-Is)

In [ ]:
national_table.to_csv(f"forecast_national_sources_{forecast_date}.csv", index=False)
region_total_table.to_csv(f"forecast_regional_totals_{forecast_date}.csv", index=False)
region_source_table.to_csv(f"forecast_regional_renewables_{forecast_date}.csv", index=False)
metrics_df.to_csv("model_evaluation_metrics_20pct.csv")
val_results.to_csv("test_predictions_20pct.csv", index=False)

print("All forecast, metrics, and validation CSV files saved successfully!")

All forecast, metrics, and validation CSV files saved successfully!


### Full Evaluation Metrics

In [ ]:
from sklearn.metrics import mean_squared_error, r2_score

metrics_list = []

for target in ALL_TARGETS:
    if target in ["wind_gen_er_mu", "wind_gen_ner_mu"]:
        continue

    # Filter the 20% test days for this target
    sub = val_results[val_results["Target_Source"] == target].copy()
    y_true = sub["Actual_MU"].values
    y_pred = sub["Predicted_MU"].values

    # 1. MAE (Mean Absolute Error in MU)
    mae = mean_absolute_error(y_true, y_pred)

    # 2. RMSE (Root Mean Squared Error in MU)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))

    # 3. WAPE % (Weighted Absolute Percentage Error)
    wape = (np.sum(np.abs(y_true - y_pred)) / (np.sum(np.abs(y_true)) + 1e-8)) * 100.0

    # 4. sMAPE % (Symmetric Mean Absolute Percentage Error)
    smape = np.mean(2.0 * np.abs(y_true - y_pred) / (np.abs(y_true) + np.abs(y_pred) + 1e-8)) * 100.0

    # 5. R2 Score
    r2 = r2_score(y_true, y_pred)

    # 6. MASE (vs. Naive "Tomorrow = Today" baseline on the exact same 20% test days)
    y_tomorrow_all = df[target].shift(-1)
    valid_mask = y_tomorrow_all.notnull() & df[target].notnull()
    split_idx = int(valid_mask.sum() * train_ratio)

    naive_test_preds = df.loc[valid_mask, target].iloc[split_idx:].values
    naive_mae = mean_absolute_error(y_true, naive_test_preds)
    mase = mae / (naive_mae + 1e-8)

    metrics_list.append({
        "Target_Source": target,
        "Strategy": "Persistence" if target in PERSISTENCE_TARGETS else "LightGBM",
        "Test_Days": len(y_true),
        "MAE_MU": round(mae, 2),
        "RMSE_MU": round(rmse, 2),
        "WAPE_%": round(wape, 2),
        "sMAPE_%": round(smape, 2),
        "R2": round(r2, 4),
        "Naive_MAE_MU": round(naive_mae, 2),
        "MASE (<1 beats Naive)": round(mase, 3),
    })

metrics_df = pd.DataFrame(metrics_list).set_index("Target_Source")

print("=== MODEL EVALUATION METRICS (20% CHRONOLOGICAL TEST SET) ===")
display(metrics_df)

=== MODEL EVALUATION METRICS (20% CHRONOLOGICAL TEST SET) ===


,Strategy,Test_Days,MAE_MU,RMSE_MU,WAPE_%,sMAPE_%,R2,Naive_MAE_MU,MASE (<1 beats Naive)
Target_Source,,,,,,,,,
hydro_gen_nr_mu,LightGBM,551,9.74,17.65,3.80,3.61,0.9783,10.03,0.971
hydro_gen_wr_mu,LightGBM,551,5.10,6.73,8.40,9.53,0.9438,5.63,0.905
hydro_gen_sr_mu,LightGBM,551,6.61,8.44,6.14,6.91,0.9633,7.67,0.862
hydro_gen_er_mu,LightGBM,551,3.35,4.83,6.17,6.86,0.9720,3.43,0.978
hydro_gen_ner_mu,LightGBM,551,1.72,2.51,5.21,5.56,0.9708,1.73,0.992
solar_gen_nr_mu,Persistence,551,16.20,23.26,7.86,8.74,0.6803,16.20,1.000
solar_gen_wr_mu,Persistence,551,10.65,15.22,6.46,7.32,0.9011,10.65,1.000
solar_gen_sr_mu,Persistence,551,11.23,14.88,7.80,8.39,0.7090,11.23,1.000
solar_gen_er_mu,Persistence,551,0.46,0.62,13.21,14.76,0.6113,0.46,1.000


In [ ]:
import pickle

# ==========================================
# SAVE PRODUCTION MODEL + ALL NEXT-DAY PREDICTED VALUES (.pkl)
# ==========================================

# 1. Ensure conformal_pads, cov_cache, and allow_neg_flags exist
if "conformal_pads" not in globals():
    conformal_pads = {}
    for target in ALL_TARGETS:
        if target in ["wind_gen_er_mu", "wind_gen_ner_mu"]:
            conformal_pads[target] = 0.0
            continue
        sub = val_results[val_results["Target_Source"] == target]
        if {"Raw_q10_MU", "Raw_q90_MU"}.issubset(sub.columns):
            cal_sub = sub[~sub["Is_Eval_Half"]] if "Is_Eval_Half" in sub.columns else sub
            nonconf = np.maximum(
                cal_sub["Raw_q10_MU"].values - cal_sub["Actual_MU"].values,
                cal_sub["Actual_MU"].values - cal_sub["Raw_q90_MU"].values,
            )
            conformal_pads[target] = float(np.quantile(nonconf, 0.80))
        else:
            conformal_pads[target] = 0.0

if "allow_neg_flags" not in globals():
    allow_neg_flags = {c: c.startswith("ie_actual_") for c in ALL_TARGETS}

if "cov_cache" not in globals():
    cal_val_df = val_results[~val_results["Is_Eval_Half"]] if "Is_Eval_Half" in val_results.columns else val_results
    cov_cache = {}
    for top_col, b_cols in [
        ("gen_total_mu", NATIONAL_SOURCES),
        ("solar_gen_total_mu", REG_SOLAR),
        ("wind_gen_total_mu", REG_WIND),
        ("energy_met_total_mu", REG_ENERGY_MET),
        ("ie_actual_total_mu", REG_IE_ACTUAL),
    ]:
        cols = [top_col] + b_cols
        E = cal_val_df[cal_val_df["Target_Source"].isin(cols)].pivot(
            index="Target_Date", columns="Target_Source", values="Error_MU"
        )[cols].dropna()
        cov_cache[top_col] = LedoitWolf().fit(E.values).covariance_

    E_hydro = cal_val_df[cal_val_df["Target_Source"].isin(REG_HYDRO)].pivot(
        index="Target_Date", columns="Target_Source", values="Error_MU"
    )[REG_HYDRO].dropna()
    cov_cache["__children__" + REG_HYDRO[0]] = LedoitWolf().fit(E_hydro.values).covariance_

if "lgb_params" not in globals():
    lgb_params = dict(
        n_estimators=250,
        learning_rate=0.02,
        num_leaves=15,
        min_child_samples=30,
        subsample=0.8,
        subsample_freq=1,
        colsample_bytree=0.7,
        reg_alpha=5.0,
        reg_lambda=10.0,
        random_state=42,
        verbose=-1,
    )

# 2. Fit final models & seasonal persistence tables on 100% of historical data
fitted_models = {}
persistence_seasonal_diffs = {}

for target in ALL_TARGETS:
    y_diff = df[target].shift(-1) - df[target]
    valid_mask = y_diff.notnull() & df[target].notnull()

    X = df.loc[valid_mask, feature_cols]
    y = y_diff.loc[valid_mask]

    if target not in PERSISTENCE_TARGETS:
        fitted_models[target] = {}
        for q, label in [(0.10, "q10"), (0.50, "q50"), (0.90, "q90")]:
            m_q = lgb.LGBMRegressor(objective="quantile", alpha=q, **lgb_params)
            m_q.fit(X, y)
            fitted_models[target][label] = m_q
    else:
        months_v = df.loc[valid_mask, "target_month"]
        p23_v = df.loc[valid_mask, "is_post_2023"]
        full_tmp = pd.DataFrame({"diff": y.values, "month": months_v.values, "p23": p23_v.values})

        monthly_bounds = {}
        for m in range(1, 13):
            sub_m = full_tmp[(full_tmp["month"] == m) & (full_tmp["p23"] == 1)]["diff"]
            if len(sub_m) < 25:
                sub_m = full_tmp[full_tmp["month"] == m]["diff"]
            d10, d90 = np.nanpercentile(sub_m, [10, 90])
            monthly_bounds[m] = (float(d10), float(d90))
        persistence_seasonal_diffs[target] = monthly_bounds

# 3. Bundle ALL next-day reconciled predicted values + models into one .pkl artifact
all_next_day_values = {
    **reconciled_preds,
    **regional_net_gen,
    **regional_gross_gen,
}

cov_col = "Cal_PostRec_Cov_%" if "Cal_PostRec_Cov_%" in metrics_df.columns else None
mae_col = "PostRec_MAE_MU" if "PostRec_MAE_MU" in metrics_df.columns else "MAE_MU"

forecaster_artifact = {
    "version": "1.1",
    "data_up_to_date": latest_date,
    "forecast_for_date": forecast_date,

    # Pre-computed next-day predicted values for ALL output columns
    "next_day_forecast": {
        "forecast_date": forecast_date,
        "all_columns_mu": all_next_day_values,
        "intervals_80pct_mu": reconciled_intervals,
        "national_sources_table": national_table,
        "regional_totals_table": region_total_table,
        "regional_renewables_table": region_source_table,
    },

    # Trained models & pipeline metadata for predicting on new data
    "regions": REGIONS,
    "reg_hydro": REG_HYDRO,
    "reg_solar": REG_SOLAR,
    "reg_wind": REG_WIND,
    "reg_energy_met": REG_ENERGY_MET,
    "reg_ie_actual": REG_IE_ACTUAL,
    "national_sources": NATIONAL_SOURCES,
    "national_totals": NATIONAL_TOTALS,
    "all_targets": ALL_TARGETS,
    "persistence_targets": PERSISTENCE_TARGETS,
    "feature_cols": feature_cols,
    "fitted_models": fitted_models,
    "persistence_seasonal_diffs": persistence_seasonal_diffs,
    "conformal_pads": conformal_pads,
    "cov_cache": cov_cache,
    "allow_neg_flags": allow_neg_flags,
    "verified_coverage": metrics_df[cov_col].to_dict() if cov_col else {},
    "post_rec_mae": metrics_df[mae_col].to_dict(),
}

artifact_path = "india_grid_forecaster.pkl"
with open(artifact_path, "wb") as f:
    pickle.dump(forecaster_artifact, f, protocol=pickle.HIGHEST_PROTOCOL)

print(f"Saved '{artifact_path}' successfully!")
print(f"Stored {len(all_next_day_values)} predicted output columns for {forecast_date}.")

Saved 'india_grid_forecaster.pkl' successfully!
Stored 46 predicted output columns for 2026-11-23.
